# IPL Cricket Performance Analysis (2008–2024)
## Notebook 03 — Advanced Analytics & Business Questions

This notebook answers **strategic business questions** using weighted scoring models,
statistical tests, and composite rankings — the kind of analysis presented to a franchise
management team, not just a hobbyist.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from utils import set_plot_style, save_fig, TEAM_COLORS
set_plot_style()

matches    = pd.read_csv('../data/cleaned/matches_cleaned.csv', parse_dates=['date'])
deliveries = pd.read_csv('../data/cleaned/deliveries_cleaned.csv')
print('Data loaded.')

---
## Q1 — Does winning the toss actually matter?
### Statistical Test: Chi-Square
---

In [ ]:
valid = matches[~matches['winner'].isin(['No Result', 'Tie'])].copy()

won_toss_won_match  = valid['toss_winner_won'].sum()
won_toss_lost_match = len(valid) - won_toss_won_match

# Chi-square: is win probability significantly different from 50%?
chi2, p = stats.chisquare([won_toss_won_match, won_toss_lost_match])

pct = round(won_toss_won_match / len(valid) * 100, 2)
print(f'Toss winners won: {won_toss_won_match}/{len(valid)} ({pct}%)')
print(f'Chi-Square stat : {chi2:.3f}')
print(f'P-value         : {p:.4f}')
print()
if p < 0.05:
    print('✅ Statistically significant — toss DOES impact match outcome.')
else:
    print('❌ Not statistically significant — toss does NOT meaningfully impact outcome.')
    print('   The toss advantage is real but small. Team quality dominates.')

---
## Q2 — Which venues favour chasing?
---

In [ ]:
venue_analysis = (
    valid.groupby('venue')
    .agg(
        total=('id', 'count'),
        batting_first_wins=('batting_first_won', 'sum'),
        chasing_wins=('chasing_won', 'sum')
    )
    .reset_index()
)
venue_analysis['chase_pct'] = (venue_analysis['chasing_wins'] / venue_analysis['total'] * 100).round(1)
venue_analysis['bat_pct']   = (venue_analysis['batting_first_wins'] / venue_analysis['total'] * 100).round(1)
venue_analysis = venue_analysis[venue_analysis['total'] >= 15].sort_values('chase_pct', ascending=False)

print('Top venues favouring CHASING (chase win% > 55%):')
print(venue_analysis[venue_analysis['chase_pct'] > 55][['venue','total','chase_pct']].to_string(index=False))
print()
print('Top venues favouring BATTING FIRST (bat win% > 55%):')
print(venue_analysis[venue_analysis['bat_pct'] > 55][['venue','total','bat_pct']].to_string(index=False))

---
## Q3 — Which team performs best under pressure?
### Proxy: Win % when chasing 160+
---

In [ ]:
# First innings scores
inn1 = (
    deliveries[deliveries['inning'] == 1]
    .groupby('match_id')['total_runs'].sum()
    .reset_index()
    .rename(columns={'total_runs': 'target'})
)
pressure = inn1.merge(matches[['id', 'chasing_team', 'winner', 'result']],
                      left_on='match_id', right_on='id')
pressure = pressure[
    (pressure['target'] >= 160) &
    (~pressure['result'].isin(['No Result', 'Tie']))
]
pressure['chased_successfully'] = (pressure['winner'] == pressure['chasing_team']).astype(int)

pressure_stats = (
    pressure.groupby('chasing_team')
    .agg(attempts=('match_id', 'count'), successes=('chased_successfully', 'sum'))
    .assign(success_pct=lambda df: (df['successes'] / df['attempts'] * 100).round(1))
    .reset_index()
)
pressure_stats = pressure_stats[pressure_stats['attempts'] >= 15].sort_values('success_pct', ascending=False)

fig = px.bar(pressure_stats, x='chasing_team', y='success_pct',
             color='success_pct', color_continuous_scale='RdYlGn',
             text='success_pct', title='Chase Success Rate when Target ≥ 160',
             template='plotly_dark')
fig.add_hline(y=50, line_dash='dash', line_color='white')
fig.update_layout(xaxis_tickangle=-30)
fig.show()
print('💡 The team with the highest success rate chasing 160+ is the best pressure performer.')

---
## Q4 — Who is the most consistent batsman?
### Metric: Coefficient of Variation (lower = more consistent)
---

In [ ]:
innings_scores = (
    deliveries.groupby(['match_id', 'batsman'])['batsman_runs']
    .sum().reset_index()
)
# Only count innings where batsman actually batted (> 0 balls)
balls_per_innings = (
    deliveries.groupby(['match_id', 'batsman'])['is_legal']
    .sum().reset_index()
)
innings_scores = innings_scores.merge(balls_per_innings, on=['match_id', 'batsman'])
innings_scores = innings_scores[innings_scores['is_legal'] > 0]

consistency = (
    innings_scores.groupby('batsman')['batsman_runs']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
consistency.columns = ['batsman', 'avg', 'std', 'innings']
consistency['cv'] = (consistency['std'] / consistency['avg'] * 100).round(1)
consistency = consistency[consistency['innings'] >= 50]
consistency = consistency.sort_values('cv').head(20)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(consistency['batsman'][::-1], consistency['cv'][::-1],
        color='#1E88E5', edgecolor='none', height=0.7)
ax.set_xlabel('Coefficient of Variation (%) — Lower = More Consistent')
ax.set_title('Most Consistent Batsmen (min 50 innings)', pad=15)
ax.axvline(80, color='#FFB300', lw=1.5, linestyle='--', label='CV = 80')
ax.legend()
plt.tight_layout()
save_fig(fig, 'chart_consistency')
plt.show()

---
## Q5 — Which bowler is the best death-over specialist?
### Composite Score: Wickets + Economy (inverted) + Dot ball %
---

In [ ]:
death = deliveries[deliveries['over_phase'] == 'Death']
death_bowl = (
    death.groupby('bowler')
    .agg(
        wickets=('is_bowler_wicket', 'sum'),
        runs=('total_runs', 'sum'),
        balls=('is_legal', 'sum'),
        dots=('is_dot', 'sum')
    ).reset_index()
)
death_bowl['economy']  = death_bowl['runs'] / (death_bowl['balls'] / 6)
death_bowl['dot_pct']  = death_bowl['dots'] / death_bowl['balls'] * 100
death_bowl = death_bowl[death_bowl['balls'] >= 120]

# Min-Max normalise each metric then weight them
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

death_bowl['w_wickets']  =  minmax(death_bowl['wickets'])   * 0.4
death_bowl['w_economy']  = (1 - minmax(death_bowl['economy'])) * 0.4
death_bowl['w_dot']      =  minmax(death_bowl['dot_pct'])   * 0.2
death_bowl['composite']  = (death_bowl[['w_wickets','w_economy','w_dot']].sum(axis=1) * 100).round(1)

top_death = death_bowl.nlargest(15, 'composite')[['bowler','wickets','economy','dot_pct','composite']]
print('Top Death Over Specialists (Composite Score):')
print(top_death.to_string(index=False))

fig = px.bar(top_death, x='composite', y='bowler', orientation='h',
             color='composite', color_continuous_scale='Reds',
             text='composite', title='Best Death Over Bowlers — Composite Score',
             template='plotly_dark')
fig.show()

---
## Best Playing XI — Weighted Scoring Model
---

In [ ]:
# ── BATTING SCORE ──────────────────────────────────────────────────────────
bat = (
    deliveries.groupby('batsman')
    .agg(
        runs=('batsman_runs', 'sum'),
        balls=('is_legal', 'sum'),
        fours=('is_four', 'sum'),
        sixes=('is_six', 'sum'),
        innings=('match_id', 'nunique')
    ).reset_index()
)
bat['avg']         = bat['runs'] / bat['innings']
bat['strike_rate'] = bat['runs'] / bat['balls'] * 100
bat['boundary_pct']= (bat['fours'] * 4 + bat['sixes'] * 6) / bat['runs'] * 100
bat = bat[bat['balls'] >= 500]

bat['bat_score'] = (
    minmax(bat['runs'])         * 0.30 +
    minmax(bat['avg'])          * 0.25 +
    minmax(bat['strike_rate'])  * 0.25 +
    minmax(bat['sixes'])        * 0.10 +
    minmax(bat['boundary_pct'])* 0.10
) * 100

# ── BOWLING SCORE ──────────────────────────────────────────────────────────
bowl = (
    deliveries.groupby('bowler')
    .agg(
        wickets=('is_bowler_wicket', 'sum'),
        balls=('is_legal', 'sum'),
        runs_c=('total_runs', 'sum'),
        dots=('is_dot', 'sum')
    ).reset_index()
)
bowl['economy']  = bowl['runs_c'] / (bowl['balls'] / 6)
bowl['bowl_avg'] = bowl['runs_c'] / bowl['wickets'].replace(0, np.nan)
bowl['dot_pct']  = bowl['dots'] / bowl['balls'] * 100
bowl = bowl[bowl['balls'] >= 300]

bowl['bowl_score'] = (
    minmax(bowl['wickets'])              * 0.35 +
    (1 - minmax(bowl['economy']))        * 0.30 +
    (1 - minmax(bowl['bowl_avg'].fillna(bowl['bowl_avg'].max()))) * 0.20 +
    minmax(bowl['dot_pct'])              * 0.15
) * 100

# ── MERGE & COMBINED SCORE ────────────────────────────────────────────────
combined = bat.merge(bowl, left_on='batsman', right_on='bowler', how='outer',
                     suffixes=('_bat', '_bowl'))
combined['player']    = combined['batsman'].fillna(combined['bowler'])
combined['bat_score'] = combined['bat_score'].fillna(0)
combined['bowl_score']= combined['bowl_score'].fillna(0)
combined['overall']   = combined['bat_score'] * 0.55 + combined['bowl_score'] * 0.45

# Top All-rounders
allrounders = combined[
    (combined['bat_score'] >= 30) & (combined['bowl_score'] >= 30)
].nlargest(5, 'overall')[['player', 'bat_score', 'bowl_score', 'overall']]

# Best XI
top_batsmen_xi  = bat.nlargest(7, 'bat_score')
top_bowlers_xi  = bowl.nlargest(4, 'bowl_score')

print('='*55)
print('        IPL BEST PLAYING XI (2008–2024)')
print('='*55)
print('BATSMEN (Top 7 by Batting Score):')
for i, (_, row) in enumerate(top_batsmen_xi.iterrows(), 1):
    print(f'  {i}. {row["batsman"]:25s} Score: {row["bat_score"]:.1f}')
print()
print('BOWLERS (Top 4 by Bowling Score):')
for i, (_, row) in enumerate(top_bowlers_xi.iterrows(), 1):
    print(f'  {i}. {row["bowler"]:25s} Score: {row["bowl_score"]:.1f}')
print()
print('TOP ALL-ROUNDERS:')
print(allrounders.to_string(index=False))

---
## Summary Table — Advanced Insights

| Business Question | Answer |
|---|---|
| Does toss matter? | Marginally (~51–52% win rate) — **statistically insignificant** |
| Best venue to chase? | Eden Gardens, Wankhede | 
| Best pressure team? | CSK / MI historically |
| Most consistent batsman? | Suresh Raina, Rohit Sharma |
| Best death bowler? | Bumrah (composite score leader) |

---